<a href="https://colab.research.google.com/github/mancinigabriel/tcc-pece-assin2-llm-challenges/blob/main/notebooks/gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/mancinigabriel/tcc-pece-assin2-llm-challenges.git

Cloning into 'tcc-pece-assin2-llm-challenges'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 136 (delta 68), reused 77 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 134.87 KiB | 5.86 MiB/s, done.
Resolving deltas: 100% (68/68), done.


#Preparando ambiente

In [2]:
import sys
import os

PROJECT_ROOT = os.path.abspath('/content/tcc-pece-assin2-llm-challenges')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src import prompts
from src import utils
from src import data
from src import metrics
import pandas as pd
import random
import torch
import re

In [5]:
cfg = utils.load_config(
    "/content/tcc-pece-assin2-llm-challenges/configs/base.yaml",
    "/content/tcc-pece-assin2-llm-challenges/configs/models/gemma.yaml"
)

generation_args = cfg["generation"]

gpu = 'A100 RAM alta'

# Gemma2 - 2b - It

In [6]:
timing = {}

timing['inicio'] = utils.time_log()

model_id = "google/gemma-2-2b-it"
model_name = "gemma_2_2b_it"
quantizado = False
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
df_assin_2 = data.gera_df()

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [7]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:07:40
100 - 2026-01-12 03:07:55
200 - 2026-01-12 03:08:11
300 - 2026-01-12 03:08:26
400 - 2026-01-12 03:08:42


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:08:57
100 - 2026-01-12 03:09:13
200 - 2026-01-12 03:09:28
300 - 2026-01-12 03:09:44
400 - 2026-01-12 03:09:59


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


...

/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:11:32
100 - 2026-01-12 03:11:48
200 - 2026-01-12 03:12:03
300 - 2026-01-12 03:12:19
400 - 2026-01-12 03:12:34


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:12:50
100 - 2026-01-12 03:13:05
200 - 2026-01-12 03:13:21
300 - 2026-01-12 03:13:36
400 - 2026-01-12 03:13:52


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))


In [8]:
next(model.parameters()).dtype

torch.bfloat16

In [9]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

0 - 2026-01-12 03:14:09
100 - 2026-01-12 03:14:25
200 - 2026-01-12 03:14:40
300 - 2026-01-12 03:14:55
400 - 2026-01-12 03:15:11
500 - 2026-01-12 03:15:26
600 - 2026-01-12 03:15:41
700 - 2026-01-12 03:15:57
800 - 2026-01-12 03:16:12
900 - 2026-01-12 03:16:27
1000 - 2026-01-12 03:16:43
1100 - 2026-01-12 03:16:58
1200 - 2026-01-12 03:17:14
1300 - 2026-01-12 03:17:29
1400 - 2026-01-12 03:17:44
1500 - 2026-01-12 03:18:00
1600 - 2026-01-12 03:18:15
1700 - 2026-01-12 03:18:31
1800 - 2026-01-12 03:18:46
1900 - 2026-01-12 03:19:01
2000 - 2026-01-12 03:19:17
2100 - 2026-01-12 03:19:32
2200 - 2026-01-12 03:19:48
2300 - 2026-01-12 03:20:03
2400 - 2026-01-12 03:20:19
2500 - 2026-01-12 03:20:34
2600 - 2026-01-12 03:20:49
2700 - 2026-01-12 03:21:05
2800 - 2026-01-12 03:21:20
2900 - 2026-01-12 03:21:36
3000 - 2026-01-12 03:21:51
3100 - 2026-01-12 03:22:06
3200 - 2026-01-12 03:22:22
3300 - 2026-01-12 03:22:37
3400 - 2026-01-12 03:22:53
3500 - 2026-01-12 03:23:08
3600 - 2026-01-12 03:23:23
3700 - 2026-0

'Arquivo salvo em /content/drive/MyDrive/Mestrado/TCC Pós/Dados/logs/log_gemma_2_2b_it_20260112_033805.json às 2026-01-12 03:38:05'

# Gemma 2 - 9B - Instruct

In [10]:
timing = {}

timing['inicio'] = utils.time_log()

model_id = "google/gemma-2-9b-it"
model_name = "gemma_2_9b_it"
quantizado = False
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
df_assin_2 = data.gera_df()

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 1331dde6-6e00-41f5-9867-6613063919c0)')' thrown while requesting GET https://huggingface.co/datasets/nilc-nlp/assin2/resolve/main/data/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


In [11]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:39:08
100 - 2026-01-12 03:39:33
200 - 2026-01-12 03:39:58
300 - 2026-01-12 03:40:23
400 - 2026-01-12 03:40:48


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:41:14
100 - 2026-01-12 03:41:39
200 - 2026-01-12 03:42:04
300 - 2026-01-12 03:42:29
400 - 2026-01-12 03:42:54


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


...

/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:45:26
100 - 2026-01-12 03:45:51
200 - 2026-01-12 03:46:16
300 - 2026-01-12 03:46:42
400 - 2026-01-12 03:47:07


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 03:47:32
100 - 2026-01-12 03:47:57
200 - 2026-01-12 03:48:22
300 - 2026-01-12 03:48:48
400 - 2026-01-12 03:49:13


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))


In [12]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

0 - 2026-01-12 03:49:38
100 - 2026-01-12 03:50:03
200 - 2026-01-12 03:50:29
300 - 2026-01-12 03:50:54
400 - 2026-01-12 03:51:19
500 - 2026-01-12 03:51:44
600 - 2026-01-12 03:52:10
700 - 2026-01-12 03:52:35
800 - 2026-01-12 03:53:00
900 - 2026-01-12 03:53:26
1000 - 2026-01-12 03:53:51
1100 - 2026-01-12 03:54:16
1200 - 2026-01-12 03:54:42
1300 - 2026-01-12 03:55:07
1400 - 2026-01-12 03:55:32
1500 - 2026-01-12 03:55:58
1600 - 2026-01-12 03:56:23
1700 - 2026-01-12 03:56:48
1800 - 2026-01-12 03:57:13
1900 - 2026-01-12 03:57:38
2000 - 2026-01-12 03:58:03
2100 - 2026-01-12 03:58:28
2200 - 2026-01-12 03:58:53
2300 - 2026-01-12 03:59:18
2400 - 2026-01-12 03:59:43
2500 - 2026-01-12 04:00:08
2600 - 2026-01-12 04:00:33
2700 - 2026-01-12 04:00:58
2800 - 2026-01-12 04:01:24
2900 - 2026-01-12 04:01:49
3000 - 2026-01-12 04:02:14
3100 - 2026-01-12 04:02:39
3200 - 2026-01-12 04:03:04
3300 - 2026-01-12 04:03:30
3400 - 2026-01-12 04:03:55
3500 - 2026-01-12 04:04:20
3600 - 2026-01-12 04:04:45
3700 - 2026-0

'Arquivo salvo em /content/drive/MyDrive/Mestrado/TCC Pós/Dados/logs/log_gemma_2_9b_it_20260112_042832.json às 2026-01-12 04:28:32'